## THIS ONLY WORKS FOR DEFAULT NEUTRAL
- Updated inpPipeline to include misalignment

# Aim for max movement of < 1.27 mm (row pitch) ?
 - Cos max pressure is in correct location so don't want to go to far
 - Also it would be visibly obvious if they were much futher off?

 ---
 - registration errors are around 0.5 mm and 0.5 degrees

In [79]:
import numpy as np
import pyvista as pv
from pathlib import Path
from phd_helpers.AbaqusPreprocessing import position_mc1_tpm
import json

from phd_helpers.paths import transform_mesh, get_bone_inertia, get_subject_stl_path, transform_points

In [ ]:
sub = '14548R'
path = Path('../../../../../Computational/MeshPipeline/outputs/initialFEAstuff/35T/35Tbest')
tpm_path = list(path.glob(f'**/{sub}/tpm-mc1/**/*.vtu'))[0]
mc1_path = list(path.glob(f'**/{sub}/mc1-tpm/**/*.vtu'))[0]

tpm = pv.read(tpm_path)
mc1 = pv.read(mc1_path)

# transform to mc1 coordinate system 
# - so that movements are relative to mc1 inertial axes - for consistency
stl_path = get_subject_stl_path(sub[:-1], sub[-1])
mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
tpm_centroid, _, _ = get_bone_inertia(stl_path, 'tpm')

tpm_mc1 = transform_mesh(tpm, mc1_axes, mc1_centroid, inverse=True)
mc1_mc1 = transform_mesh(mc1, mc1_axes, mc1_centroid, inverse=True)
tpm_centroid = transform_points(tpm_centroid, mc1_axes, mc1_centroid, inverse=True)[0]

In [93]:
R = 5 # degrees
t = 1.0 # mm
tpm_Rx = tpm_mc1.rotate_x(R, point=tpm_centroid, transform_all_input_vectors=True)
tpm_Ry = tpm_mc1.rotate_y(R, point=tpm_centroid, transform_all_input_vectors=True)
tpm_Rz = tpm_mc1.rotate_z(R, point=tpm_centroid, transform_all_input_vectors=True)

tpm_ty = tpm_mc1.translate([0., t, 0.], transform_all_input_vectors=True)
tpm_tz = tpm_mc1.translate([0., 0., t], transform_all_input_vectors=True)

# max point movement
np.linalg.norm(tpm_mc1.points - tpm_Rx.points, axis=1).max()

np.float64(0.9869448595145695)

In [96]:
mesh = tpm_Rx

pl = pv.Plotter()
pl.add_mesh(mc1_mc1, color='white')
pl.add_mesh(tpm_mc1, color='black', style='wireframe')
pl.add_mesh(mesh, color='white')
pl.show_axes()
pl.show()

Widget(value='<iframe src="http://localhost:54329/index.html?ui=P_0x373377710_30&reconnect=auto" class="pyvist…

In [95]:
# transform back to global

mesh_glob = transform_mesh(mesh, mc1_axes, mc1_centroid)

pl = pv.Plotter()
pl.add_mesh(mc1, color='white')
pl.add_mesh(tpm, color='black', style='wireframe')
pl.add_mesh(mesh_glob, color='white')
pl.show_axes()
pl.show()

Widget(value='<iframe src="http://localhost:54329/index.html?ui=P_0x373389e20_29&reconnect=auto" class="pyvist…

In [97]:
# save meshes in global 

tpm_savedir = path.parent / f'TwistTranslate1/meshes/{sub}/tpm-mc1/3Dmesh'
tpm_savedir.mkdir(parents=True, exist_ok=True)

mc1_savedir = path.parent / f'TwistTranslate1/meshes/{sub}/mc1-tpm/3Dmesh'
mc1_savedir.mkdir(parents=True, exist_ok=True)

names = ['Rx', 'Ry', 'Rz', 'ty', 'tz']
meshes = [tpm_Rx, tpm_Ry, tpm_Rz, tpm_ty, tpm_tz]

for mesh, name in zip(meshes, names):

    mesh_glob = transform_mesh(mesh, mc1_axes, mc1_centroid)
    mesh_glob.save(tpm_savedir / f'mesh-{name}.vtu')
    mc1.save(mc1_savedir / f'mesh-{name}.vtu')

# Verify .inp meshes are correct
 - They get translated along x slightly for inp file so just check other alignements

In [98]:
from phd_helpers.AbaqusPostprocessing import inp2pv

In [ ]:
inp_path = Path('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/study1')
name = 'ty'
mesh = tpm_ty

inp_mesh = inp2pv(list(inp_path.glob(f'**/{name}*.inp'))[0])
inp_tpm = inp_mesh['tpm']
inp_mc1 = inp_mesh['mc1']

In [120]:
pl = pv.Plotter()
pl.add_mesh(inp_mc1, color='white')
pl.add_mesh(inp_tpm, color='white')
pl.add_mesh(mesh, color='black', style='wireframe')
pl.show_axes()
pl.show()

Widget(value='<iframe src="http://localhost:54329/index.html?ui=P_0x3732cd520_36&reconnect=auto" class="pyvist…

# Study 2
 - Negatives of study 1

In [121]:
import numpy as np
import pyvista as pv
from pathlib import Path

from phd_helpers.paths import transform_mesh, get_bone_inertia, get_subject_stl_path, transform_points

In [123]:
R = -5 # degrees
t = -1.0 # mm

sub = '14548R'
mesh_out_root_name = 'TwistTranslate2'

In [ ]:
# load meshes
path = Path('../../../../../Computational/MeshPipeline/outputs/initialFEAstuff/35T/35Tbest')
tpm_path = list(path.glob(f'**/{sub}/tpm-mc1/**/*.vtu'))[0]
mc1_path = list(path.glob(f'**/{sub}/mc1-tpm/**/*.vtu'))[0]

tpm = pv.read(tpm_path)
mc1 = pv.read(mc1_path)

# transform to mc1 coordinate system 
# - so that movements are relative to mc1 inertial axes - for consistency
stl_path = get_subject_stl_path(sub[:-1], sub[-1])
mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
tpm_centroid, _, _ = get_bone_inertia(stl_path, 'tpm')

tpm_mc1 = transform_mesh(tpm, mc1_axes, mc1_centroid, inverse=True)
mc1_mc1 = transform_mesh(mc1, mc1_axes, mc1_centroid, inverse=True)
tpm_centroid = transform_points(tpm_centroid, mc1_axes, mc1_centroid, inverse=True)[0]

tpm_Rx = tpm_mc1.rotate_x(R, point=tpm_centroid, transform_all_input_vectors=True)
tpm_Ry = tpm_mc1.rotate_y(R, point=tpm_centroid, transform_all_input_vectors=True)
tpm_Rz = tpm_mc1.rotate_z(R, point=tpm_centroid, transform_all_input_vectors=True)

tpm_ty = tpm_mc1.translate([0., t, 0.], transform_all_input_vectors=True)
tpm_tz = tpm_mc1.translate([0., 0., t], transform_all_input_vectors=True)

# save meshes in global 
tpm_savedir = path.parent / f'{mesh_out_root_name}/meshes/{sub}/tpm-mc1/3Dmesh'
tpm_savedir.mkdir(parents=True, exist_ok=True)

mc1_savedir = path.parent / f'{mesh_out_root_name}/meshes/{sub}/mc1-tpm/3Dmesh'
mc1_savedir.mkdir(parents=True, exist_ok=True)

names = ['Rx', 'Ry', 'Rz', 'ty', 'tz']
meshes = [tpm_Rx, tpm_Ry, tpm_Rz, tpm_ty, tpm_tz]

for mesh, name in zip(meshes, names):

    mesh_glob = transform_mesh(mesh, mc1_axes, mc1_centroid)
    mesh_glob.save(tpm_savedir / f'mesh-{name}.vtu')
    mc1.save(mc1_savedir / f'mesh-{name}.vtu')



# 20260904 - updated inpPipeline to include misalignment
 - Replicate real instron misalignment

In [23]:
def load_and_transform_meshes(sub, path):
    """Returns tpm and mc1 in mc1 inertial coords and returns tpm_centroid for centre of rotation"""
    tpm_path = list(path.glob(f'**/{sub}/tpm-mc1/**/*.vtu'))[0]
    mc1_path = list(path.glob(f'**/{sub}/mc1-tpm/**/*.vtu'))[0]

    tpm = pv.read(tpm_path)
    mc1 = pv.read(mc1_path)

    # transform to mc1 coordinate system 
    # - so that movements are relative to mc1 inertial axes - for consistency
    stl_path = get_subject_stl_path(sub[:-1], sub[-1])
    mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
    tpm_centroid, _, _ = get_bone_inertia(stl_path, 'tpm')

    tpm_mc1 = transform_mesh(tpm, mc1_axes, mc1_centroid, inverse=True)
    mc1_mc1 = transform_mesh(mc1, mc1_axes, mc1_centroid, inverse=True)
    tpm_centroid = transform_points(tpm_centroid, mc1_axes, mc1_centroid, inverse=True)[0]

    return tpm_mc1, mc1_mc1, tpm_centroid

def rotate_mesh(mesh1, cor, Rx=0.0, Ry=0.0, Rz=0.0):
    """Centre of Rotation (cor)"""
    mesh = mesh1.copy()
    mesh.rotate_x(Rx, point=cor, inplace=True, transform_all_input_vectors=True)
    mesh.rotate_y(Ry, point=cor, inplace=True, transform_all_input_vectors=True)
    mesh.rotate_z(Rz, point=cor, inplace=True, transform_all_input_vectors=True)
    return mesh

def translate_mesh(mesh1, tx=0.0, ty=0.0, tz=0.0):
    mesh = mesh1.copy()
    mesh.translate([tx, ty, tz], inplace=True, transform_all_input_vectors=True)
    return mesh

### trapezium needs translating slightly along the y-axis in the positive direction
 - based on plot below and images from 07/07/26 (instron alignemnt) and 04/09/26 (parts in instron)
 - comparing below plot with parts in inston image, it looks like misalignment is between 0.5 and 1mm

In [148]:
sub = '14548R'
path = Path('../../../../../Computational/MeshPipeline/outputs/initialFEAstuff/35T/35Tbest')
tpm_mc1, mc1_mc1, tpm_centroid = load_and_transform_meshes(sub, path)

# following transforms and positioning replicate what happens during .inp creation
tx, ty = -3, 0.75
tpm_mc1_t = rotate_mesh(tpm_mc1, tpm_centroid, Rz=0)
tpm_mc1_t = translate_mesh(tpm_mc1_t, ty=0.75, tx=-3)

#position_mc1_tpm(mc1_mc1, tpm_mc1_t, target_dist=0.01)
position_mc1_tpm(mc1_mc1, tpm_mc1, target_dist=0.01)

# max point movement
print(round(np.linalg.norm(tpm_mc1.points - tpm_mc1_t.points, axis=1).max(), 2))

Distance between cartilage surfaces (x->): 0.0100
No interference:  True
3.14


In [149]:
pl = pv.Plotter()
pl.add_mesh(mc1_mc1, color='white')
pl.add_mesh(tpm_mc1_t, color='white')
#pl.add_mesh(tpm_mc1_t, color='black', style='wireframe')
pl.camera_position = pv.CameraPosition(position=(-20.132066950629557, 7.900546633102784, 77.40407765924168),
               focal_point=(-20.740850107506702, 5.098509843538982, 4.131159924631769),
               viewup=(-0.9999652287891094, 0.0011020039560094039, 0.008266002663034832))
pl.add_axes()
pl.show()

Widget(value='<iframe src="http://localhost:59770/index.html?ui=P_0x387607410_62&reconnect=auto" class="pyvist…

#### Verify .inp mesh is correct
- After generating .inp files

In [76]:
from phd_helpers.AbaqusPostprocessing import inp2pv
inp_root = Path('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/test_integration')
inp_paths = inp_root.glob('**/*.inp')
list(inp_paths)

[PosixPath('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/test_integration/inpFiles/50017L/inp/35T-flexion-04/35T-flexion-04.inp'),
 PosixPath('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/test_integration/inpFiles/50017L/inp/35T-flexion-03/35T-flexion-03.inp'),
 PosixPath('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/test_integration/inpFiles/50017L/inp/35T-flexion-02/35T-flexion-02.inp'),
 PosixPath('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/test_integration/inpFiles/50017L/inp/35T-flexion-05/35T-flexion-05.inp'),
 PosixPath('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/test_integration/inpFiles/50017L/inp/35T-neutral-00/35T-neutral-00.inp'),
 PosixPath('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/test_integration/inpFile

In [152]:
from phd_helpers.paths import get_relative_transform_new_basis, get_bone_transforms, get_subject_stl_path, pose2idCMC, transform_mesh

In [172]:
sub = '50017L'
pose = 'flexion'
run_id = '03'
inp_path = list(inp_root.glob(f'**/{sub}/**/*-{pose}-{run_id}.inp'))[0]

# get misalignment (and target dist to position original meshes)
loop_param_path = inp_path.parents[4] / f'params/loop_params/{run_id}.json'
with open(loop_param_path, 'r') as f:
    run_params = json.load(f)

target_dist = run_params['target_dist']
misalign_R = run_params['misalign_R']
misalign_t = run_params['misalign_t']
print('R', misalign_R)
print('t', misalign_t)

# original mesh in neutral
mesh_path = Path('../../../../../Computational/MeshPipeline/outputs/initialFEAstuff/35T/35Tbest')
tpm_mc1, mc1_mc1, _ = load_and_transform_meshes(sub, path)
if pose != 'neutral':
    stl_path = get_subject_stl_path(sub[:-1], sub[-1])
    mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
    transforms = get_bone_transforms(pose2idCMC(pose), stl_path)
    R, t = get_relative_transform_new_basis(transforms, 'tpm', 'mc1', mc1_centroid, mc1_axes)
    tpm_mc1 = transform_mesh(tpm_mc1, R, t)
position_mc1_tpm(mc1_mc1, tpm_mc1, target_dist=target_dist)

# inp mesh
inp_mesh = inp2pv(inp_path)
inp_tpm = inp_mesh['tpm']
inp_mc1 = inp_mesh['mc1']



pl = pv.Plotter()
pl.add_mesh(mc1_mc1, color='white')
pl.add_mesh(tpm_mc1, color='white')
pl.add_mesh(inp_tpm, color='black', style='wireframe')
pl.add_mesh(inp_mc1, color='black', style='wireframe')
pl.camera_position = pv.CameraPosition(position=(-20.132066950629557, 7.900546633102784, 77.40407765924168),
               focal_point=(-20.740850107506702, 5.098509843538982, 4.131159924631769),
               viewup=(-0.9999652287891094, 0.0011020039560094039, 0.008266002663034832))
pl.show_axes()
pl.show()

R [0, 0, 0]
t [0, 1, 0]
Distance between cartilage surfaces (x->): 0.0100
No interference:  True


Widget(value='<iframe src="http://localhost:59770/index.html?ui=P_0x33b975f70_71&reconnect=auto" class="pyvist…

In [122]:
for run_id in ['00', '01', '02', '03', '04', '05']:
    loop_param_path = inp_path.parents[4] / f'params/loop_params/{run_id}.json'
    with open(loop_param_path, 'r') as f:
        run_params = json.load(f)

    misalign_R = run_params['misalign_R']
    misalign_t = run_params['misalign_t']
    print(run_id, 'R', misalign_R)
    print('  ', 't', misalign_t)

00 R [0, 0, 0]
   t [0, 0, 0]
01 R [5, 0, 0]
   t [0, 0, 0]
02 R [0, 0, 5]
   t [0, 0, 0]
03 R [0, 0, 0]
   t [0, 1, 0]
04 R [5, 0, 0]
   t [0, 1, 0]
05 R [0, 0, 5]
   t [0, 1, 0]
